# Lanjut fine-tuning Dataset Pribadi dari backup depth
Salin notebook ke akun tujuan. Sediakan di My Drive akun tujuan: `CDCN-PRNet-Depth-Colab.zip`, `CDCN_Depth/pribadi_prepared.zip`, dan `CDCN_Depth/siw/best.pth`. File boleh berupa salinan atau pintasan ke file yang dibagikan. Aktifkan runtime GPU, lalu jalankan sel berurutan. Notebook ini memakai depth yang sudah tersedia dan tidak menjalankan PRNet lagi.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, subprocess, zipfile
import torch

drive.mount('/content/drive')
print('GPU tersedia:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU belum tersedia. Hentikan di sini sampai GPU tersedia.'
print('GPU:', torch.cuda.get_device_name(0))
BASE = Path('/content/drive/MyDrive')
SOURCE = BASE / 'CDCN-PRNet-Depth-Colab.zip'
BACKUP = BASE / 'CDCN_Depth/pribadi_prepared.zip'
CHECKPOINT = BASE / 'CDCN_Depth/siw/best.pth'
if not BACKUP.is_file():
    for candidate in (BASE / "CDCN_Depth").glob("*_prepared.zip"):
        with zipfile.ZipFile(candidate) as archive:
            if "train_only_depth.csv" in archive.namelist():
                BACKUP = candidate
                break

for p in (SOURCE, BACKUP, CHECKPOINT):
    print(p, 'ditemukan:', p.is_file())
    assert p.is_file(), f'File belum tersedia: {p}'


In [ ]:
ROOT = Path('/content/cdcn-depth')
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SOURCE) as archive:
    archive.extractall(ROOT)

# Normalize a previous private archive's dataset directory by its manifests.
data_parent = ROOT / "data"
private_root = data_parent / "pribadi"
if not private_root.exists() and data_parent.exists():
    candidates = [p for p in data_parent.iterdir()
                  if p.is_dir() and (p / "train_only.csv").is_file()]
    if len(candidates) == 1:
        candidates[0].rename(private_root)

with zipfile.ZipFile(BACKUP) as archive:
    archive.extractall(ROOT / 'data/pribadi')
os.chdir(ROOT)

def run(*args):
    with subprocess.Popen([str(a) for a in args], cwd=ROOT,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        result = process.wait()
    if result:
        raise RuntimeError('Perintah gagal. Periksa pesan di atas.')

run(sys.executable, '-m', 'pip', 'install', '-q',
    'tensorboard', 'pyyaml', 'opencv-python-headless', 'matplotlib')
print('Kode dan depth Dataset Pribadi sudah dipulihkan.')


In [ ]:
state = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
print('Epoch checkpoint SiW:', state.get('epoch', -1) + 1)
print('Threshold SiW:', state.get('threshold'))
del state
summary = ROOT / 'data/pribadi/depth_summary.json'
if summary.is_file():
    print(summary.read_text())
for variant in ('only', 'mixed'):
    run(sys.executable, '-u', 'train.py', '--config',
        f'config/CDCN_depth_pribadi_{variant}.yaml', '--check-data')
print('Pemeriksaan data selesai. Lima kegagalan deteksi tidak dimasukkan ke manifest depth.')


## Fine-tuning dataset pribadi saja
Maksimum 50 epoch. Memulai dari checkpoint depth SiW; jika latest.pth adaptasi sudah ada, melanjutkan checkpoint tersebut. Threshold dipilih dari validasi Dataset Pribadi.


In [ ]:
variant = 'only'
latest = BASE / f'CDCN_Depth/pribadi_{variant}/latest.pth'
command = [sys.executable, '-u', 'train.py', '--config',
           f'config/CDCN_depth_pribadi_{variant}.yaml']
command += ['--resume', str(latest)] if latest.is_file() else ['--init-checkpoint', str(CHECKPOINT)]
run(*command)


## Fine-tuning dataset pribadi gabungan
Jalankan setelah dataset pribadi saja selesai. Run pertama juga dimulai dari SiW yang sama. Lima gambar spoof gagal deteksi dikecualikan; training memakai 679 gambar jika backup sesuai ringkasan terakhir.


In [ ]:
variant = 'mixed'
latest = BASE / f'CDCN_Depth/pribadi_{variant}/latest.pth'
command = [sys.executable, '-u', 'train.py', '--config',
           f'config/CDCN_depth_pribadi_{variant}.yaml']
command += ['--resume', str(latest)] if latest.is_file() else ['--init-checkpoint', str(CHECKPOINT)]
run(*command)


## Evaluasi test setelah kedua fine-tuning selesai
Gunakan threshold dari validasi masing-masing checkpoint. Jangan memilih threshold berdasarkan hasil test.


In [ ]:
for variant in ('only', 'mixed'):
    print('\nTest', variant, flush=True)
    run(sys.executable, '-u', 'evaluate.py', '--config',
        f'config/CDCN_depth_pribadi_{variant}.yaml',
        '--checkpoint', str(BASE / f'CDCN_Depth/pribadi_{variant}/best.pth'),
        '--test-csv', 'test_depth.csv')
